# 04_train_model_B — Train efficientnet_b2

Train a pretrained **efficientnet_b2** with transfer learning. Crash-resilient:

- Every epoch: training history JSON + epoch CSV → Drive
- Every best val_acc improvement: full training state checkpoint → Drive
- Resume support: if a checkpoint exists, set `RESUME = True` to continue

Output:
- `results/metrics/efficientnet_b2_model_card.{md,json}` (committed to GitHub)
- `pk_politicians_results/checkpoints/efficientnet_b2_best.pth` (Drive)
- `pk_politicians_results/logs/efficientnet_b2_history.json` (Drive)
- `pk_politicians_results/logs/efficientnet_b2_epoch_log.csv` (Drive)

## 1. Environment setup

This notebook is designed for **Google Colab (A100)**. The setup cell below:

1. Mounts Google Drive
2. Clones or updates the GitHub repo
3. Installs requirements
4. Adds the repo root to `sys.path` so `from src...` works

In [11]:
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/musarashid49/Image-Classification-with-CNN.git"
REPO_DIR = Path("/content/Image-Classification-with-CNN")

# 1. Mount Drive
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print("Not in Colab — skipping drive mount.")

# 2. Clone or pull
if REPO_DIR.exists():
    print(f"Repo already cloned at {REPO_DIR}; pulling latest...")
    subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=False)
else:
    print(f"Cloning {REPO_URL} -> {REPO_DIR}")
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

# 3. Make repo importable
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

# 4. Install requirements (Colab usually has torch already)
req = REPO_DIR / "requirements.txt"
if req.exists():
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(req)], check=False)

os.chdir(REPO_DIR)
print(f"cwd: {os.getcwd()}")
print(f"sys.path[0]: {sys.path[0]}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Repo already cloned at /content/Image-Classification-with-CNN; pulling latest...
cwd: /content/Image-Classification-with-CNN
sys.path[0]: /content/Image-Classification-with-CNN


## 2. Config — edit here to switch model or hyperparameters

In [12]:
# ---- Model & training config (single source of truth for this notebook) ----
MODEL_NAME = "efficientnet_b2"        # see src/models.py for options
EPOCHS = 35
LR_HEAD = 0.001
LR_BACKBONE = 5e-05
WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.05
EARLY_STOPPING_PATIENCE = 8
SCHEDULER_PATIENCE = 3
DROPOUT_HEAD = 0.2
BATCH_SIZE = 32
RESUME = False           # set True to resume from existing checkpoint on Drive
FORCE_RETRAIN = False    # set True to ignore an existing _best checkpoint and retrain

## 3. Imports & setup

In [13]:
import time
import torch
from dataclasses import asdict

from config.config import (
    CLASS_NAMES, DATASET_DIR,
    LOCAL_RESULTS, LOCAL_CHECKPOINTS, LOCAL_METRICS,
    RESULTS_DRIVE, CHECKPOINTS_DRIVE,
    SEED, IMG_SIZE,
)
from src.utils import (
    set_seed, get_device, gpu_info, ensure_dir,
    save_model_card, ExperimentLogger,
)
from src.dataset import build_dataloaders
from src.models import build_model, get_param_groups
from src.train import Trainer, TrainConfig

set_seed(SEED)
device = get_device()
print(f"Device: {device}")
print(f"GPU: {gpu_info() or 'CPU only'}")

Device: cuda
GPU: NVIDIA A100-SXM4-40GB (42.4 GB)


## 4. Data loaders

In [ ]:
train_loader, val_loader, test_loader, class_to_idx = build_dataloaders(
    dataset_root=DATASET_DIR,
    batch_size=BATCH_SIZE,
)
print(f"train batches: {len(train_loader)}, "
      f"val batches: {len(val_loader)}, "
      f"test batches: {len(test_loader)}")
print(f"class_to_idx: {class_to_idx}")

train batches: 34, val batches: 7, test batches: 5
class_to_idx: {'ahmed_sharif_chaudhry': 0, 'altaf_hussain': 1, 'asfandyar_wali': 2, 'asif_ali_zardari': 3, 'bilawal_bhutto': 4, 'chaudhry_nisar': 5, 'fazlur_rehman': 6, 'imran_khan': 7, 'maryam_nawaz': 8, 'nawaz_sharif': 9, 'pervez_khattak': 10, 'pervez_musharraf': 11, 'rana_sanaullah': 12, 'shah_mehmood_qureshi': 13, 'shehbaz_sharif': 14, 'sirajul_haq': 15}


## 5. Build model

In [15]:
model, info = build_model(MODEL_NAME, dropout=DROPOUT_HEAD)
param_groups = get_param_groups(
    model,
    lr_head=LR_HEAD, lr_backbone=LR_BACKBONE,
    weight_decay=WEIGHT_DECAY,
)
print(f"Model: {info['model_name']}")
print(f"Total params: {info['total_params']:,} ({info['total_params_M']}M)")
print(f"Trainable: {info['trainable_params']:,}")

Model: efficientnet_b2
Total params: 7,723,538 (7.72M)
Trainable: 7,723,538


## 6. Resume / skip logic

In [16]:
ckpt_path_drive = CHECKPOINTS_DRIVE / f"{MODEL_NAME}_best.pth"
resume_from = None

if ckpt_path_drive.exists():
    print(f"Existing checkpoint found at {ckpt_path_drive}")
    if FORCE_RETRAIN:
        print("FORCE_RETRAIN=True — will retrain from scratch.")
    elif RESUME:
        print("RESUME=True — continuing training from this checkpoint.")
        resume_from = ckpt_path_drive
    else:
        print("Neither RESUME nor FORCE_RETRAIN is True. The trainer will only")
        print("overwrite the checkpoint if a new epoch beats the current val_acc.")
        print("If the existing run is complete, set RESUME=False, run, and watch.")

Existing checkpoint found at /content/drive/MyDrive/pk_politicians_results/checkpoints/efficientnet_b2_best.pth
Neither RESUME nor FORCE_RETRAIN is True. The trainer will only
overwrite the checkpoint if a new epoch beats the current val_acc.
If the existing run is complete, set RESUME=False, run, and watch.


## 7. Trainer config + run

In [17]:
cfg = TrainConfig(
    model_name=MODEL_NAME,
    epochs=EPOCHS,
    lr_head=LR_HEAD,
    lr_backbone=LR_BACKBONE,
    weight_decay=WEIGHT_DECAY,
    label_smoothing=LABEL_SMOOTHING,
    early_stopping_patience=EARLY_STOPPING_PATIENCE,
    scheduler_patience=SCHEDULER_PATIENCE,
)

trainer = Trainer(
    model=model,
    train_loader=train_loader, val_loader=val_loader,
    device=device, cfg=cfg,
    param_groups=param_groups,
    local_results_dir=LOCAL_RESULTS,
    drive_results_dir=RESULTS_DRIVE,
    resume_from=resume_from,
)
run = trainer.run()
print(f"\nTraining complete. best_val_acc={run['best_val_acc']:.4f} "
      f"in {run['duration_seconds']:.1f}s "
      f"({run['epochs_completed']} epochs)")

[trainer] starting efficientnet_b2 on cuda


train 1:   0%|          | 0/34 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


val   1:   0%|          | 0/7 [00:00<?, ?it/s]

[epoch 1/35] epoch=1 | train_loss=2.4247 | train_acc=0.3139 | val_loss=1.9903 | val_acc=0.6293 | lr_head=0.0010 | lr_backbone=0.0001 | epoch_seconds=11.9500
  ↑ new best val_acc=0.6293 — saved to Drive: efficientnet_b2_best.pth


...

val   26:   0%|          | 0/7 [00:00<?, ?it/s]

[epoch 26/35] epoch=26 | train_loss=0.3566 | train_acc=0.9972 | val_loss=0.6715 | val_acc=0.8927 | lr_head=0.0003 | lr_backbone=0.0000 | epoch_seconds=11.0200
  no improvement (best=0.9024, patience 7/8)


train 27:   0%|          | 0/34 [00:00<?, ?it/s]

val   27:   0%|          | 0/7 [00:00<?, ?it/s]

[epoch 27/35] epoch=27 | train_loss=0.3618 | train_acc=0.9972 | val_loss=0.6778 | val_acc=0.8927 | lr_head=0.0001 | lr_backbone=0.0000 | epoch_seconds=11.2100
  no improvement (best=0.9024, patience 8/8)
[trainer] early stopping at epoch 27
[trainer] done. best_val_acc=0.9024 in 329.3s

Training complete. best_val_acc=0.9024 in 329.3s (27 epochs)


## 8. Quick test-set evaluation + model card

In [18]:
from src.evaluate import (
    evaluate_and_save, predict_on_loader, compute_metrics
)
from config.config import LOCAL_PLOTS, REPORT_FIGURES, LOCAL_METRICS

# Load best weights (in case last epoch wasn't best)
from src.train import load_model_for_inference
model = load_model_for_inference(model, trainer.best_path_local, device)

metrics = evaluate_and_save(
    model=model,
    test_loader=test_loader,
    device=device,
    class_names=CLASS_NAMES,
    model_name=MODEL_NAME,
    plots_dir=LOCAL_PLOTS,
    report_dir=REPORT_FIGURES,
    metrics_dir=LOCAL_METRICS,
    history=trainer.history,
)
print(f"\nTest metrics for {MODEL_NAME}:")
print(f"  accuracy:        {metrics['accuracy']:.4f}")
print(f"  macro precision: {metrics['macro_precision']:.4f}")
print(f"  macro recall:    {metrics['macro_recall']:.4f}")
print(f"  macro F1:        {metrics['macro_f1']:.4f}")
print(f"  weighted F1:     {metrics['weighted_f1']:.4f}")


Test metrics for efficientnet_b2:
  accuracy:        0.8553
  macro precision: 0.8638
  macro recall:    0.8592
  macro F1:        0.8540
  weighted F1:     0.8524


In [19]:
# ── Save model card and experiment log to Drive (local persistence) ─────────
# Saved to Drive directly — push to GitHub manually when you're ready.
from config.config import RESULTS_DRIVE

METRICS_DRIVE = RESULTS_DRIVE / "metrics"
METRICS_DRIVE.mkdir(parents=True, exist_ok=True)

card_paths = save_model_card(
    model_name=MODEL_NAME,
    model_info=info,
    training_config=asdict(cfg),
    final_metrics={
        "test_accuracy":          metrics["accuracy"],
        "test_macro_precision":   metrics["macro_precision"],
        "test_macro_recall":      metrics["macro_recall"],
        "test_macro_f1":          metrics["macro_f1"],
        "test_weighted_f1":       metrics["weighted_f1"],
        "best_val_acc":           run["best_val_acc"],
        "epochs_completed":       run["epochs_completed"],
    },
    training_duration_seconds=run["duration_seconds"],
    checkpoint_drive_path=str(trainer.best_path_drive),
    output_dir=METRICS_DRIVE,                   # ← Drive, not repo
    notebook_name="04_train_model_B.ipynb",
    notes="Trained on dataset_resplit (75/15/10 split, per-class).",
)
print(f"✓ model card (md):   {card_paths['md']}")
print(f"✓ model card (json): {card_paths['json']}")

# Experiment log — also on Drive
ExperimentLogger(METRICS_DRIVE / "experiment_log.jsonl").log({
    "model":            MODEL_NAME,
    "test_accuracy":    metrics["accuracy"],
    "macro_f1":         metrics["macro_f1"],
    "epochs":           run["epochs_completed"],
    "duration_seconds": run["duration_seconds"],
})
print(f"✓ experiment log appended: {METRICS_DRIVE / 'experiment_log.jsonl'}")

print()
print("When ready to commit these to GitHub, copy them into results/metrics/ and push:")
print(f"  cp {METRICS_DRIVE}/*.json  /content/Image-Classification-with-CNN/results/metrics/")
print(f"  cp {METRICS_DRIVE}/*.md    /content/Image-Classification-with-CNN/results/metrics/")

✓ model card (md):   /content/drive/MyDrive/pk_politicians_results/metrics/efficientnet_b2_model_card.md
✓ model card (json): /content/drive/MyDrive/pk_politicians_results/metrics/efficientnet_b2_model_card.json
✓ experiment log appended: /content/drive/MyDrive/pk_politicians_results/metrics/experiment_log.jsonl

When ready to commit these to GitHub, copy them into results/metrics/ and push:
  cp /content/drive/MyDrive/pk_politicians_results/metrics/*.json  /content/Image-Classification-with-CNN/results/metrics/
  cp /content/drive/MyDrive/pk_politicians_results/metrics/*.md    /content/Image-Classification-with-CNN/results/metrics/


In [ ]:

import os, sys, json, subprocess
from pathlib import Path

# ── 1. Mount Drive + clone repo (skip if already done) ──────────────────────
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except ImportError:
    pass  # not in Colab

REPO_DIR = Path("/content/Image-Classification-with-CNN")
REPO_URL = "https://github.com/musarashid49/Image-Classification-with-CNN.git"
if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=False)
else:
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
os.chdir(REPO_DIR)

# ── 2. Imports ───────────────────────────────────────────────────────────────
import torch
import numpy as np
from config.config import (
    CLASS_NAMES, DATASET_DIR, RESULTS_DRIVE, SEED,
)
from src.utils import set_seed, get_device, gpu_info, ensure_dir, write_json
from src.dataset import build_dataloaders
from src.models import build_model
from src.train import load_model_for_inference
from src.evaluate import (
    predict_on_loader,
    compute_metrics,
    save_classification_report,
    plot_training_curves,
    plot_confusion_matrix,
    plot_per_class_metrics,
    plot_top_misclassified,
)

set_seed(SEED)
device = get_device()
print(f"Device : {device}")
print(f"GPU    : {gpu_info() or 'CPU only'}")

# ── 3. Paths on Drive ────────────────────────────────────────────────────────
MODEL_NAME    = "efficientnet_b2"
CKPT_DRIVE    = RESULTS_DRIVE / "checkpoints" / f"{MODEL_NAME}_best.pth"
HISTORY_DRIVE = RESULTS_DRIVE / "logs"        / f"{MODEL_NAME}_history.json"
PLOTS_DRIVE   = ensure_dir(RESULTS_DRIVE / "plots")
REPORT_DRIVE  = ensure_dir(RESULTS_DRIVE / "report_figures")
METRICS_DRIVE = ensure_dir(RESULTS_DRIVE / "metrics")

print(f"\nCheckpoint : {CKPT_DRIVE}")
print(f"Exists     : {CKPT_DRIVE.exists()}")
if not CKPT_DRIVE.exists():
    raise FileNotFoundError(
        f"Checkpoint not found at {CKPT_DRIVE}\n"
        "Run the training notebook first, or check the Drive path."
    )

# ── 4. Load model from checkpoint ───────────────────────────────────────────
print("\nBuilding model architecture...")
model, info = build_model(MODEL_NAME, num_classes=len(CLASS_NAMES), dropout=0.2)
print(f"  {info['model_name']}  |  {info['total_params_M']}M params")

model = load_model_for_inference(model, CKPT_DRIVE, device)
print(f"✓ Weights loaded from Drive checkpoint")

# ── 5. Load test data ────────────────────────────────────────────────────────
print("\nLoading test dataset...")
_, _, test_loader, class_to_idx = build_dataloaders(
    dataset_root=DATASET_DIR,
    batch_size=32,
)
print(f"  {len(test_loader)} test batches  |  "
      f"{len(test_loader.dataset)} test images  |  "
      f"{len(CLASS_NAMES)} classes")

# ── 6. Run inference ─────────────────────────────────────────────────────────
print("\nRunning inference on test set...")
y_true, y_pred, y_prob = predict_on_loader(model, test_loader, device)
metrics = compute_metrics(y_true, y_pred, CLASS_NAMES)

print(f"\n── Test metrics for {MODEL_NAME} ──────────────────")
print(f"  accuracy        : {metrics['accuracy']:.4f}")
print(f"  macro precision : {metrics['macro_precision']:.4f}")
print(f"  macro recall    : {metrics['macro_recall']:.4f}")
print(f"  macro F1        : {metrics['macro_f1']:.4f}")
print(f"  weighted F1     : {metrics['weighted_f1']:.4f}")
print("────────────────────────────────────────────────────")

# ── 7. Load training history for curves ─────────────────────────────────────
history = None
if HISTORY_DRIVE.exists():
    with open(HISTORY_DRIVE) as f:
        history = json.load(f)
    print(f"\n✓ Training history loaded  ({len(history.get('train_loss',[]))} epochs)")
else:
    print(f"\n⚠️  History not found at {HISTORY_DRIVE} — training curves will be skipped")

# ── 8a. Training curves ──────────────────────────────────────────────────────
print("\n[1/5] Training curves...")
if history and len(history.get("train_loss", [])) > 0:
    paths = plot_training_curves(history, MODEL_NAME, PLOTS_DRIVE, REPORT_DRIVE)
    print(f"  ✓ PNG : {PLOTS_DRIVE}/{MODEL_NAME}_training_curves.png")
    print(f"  ✓ PDF : {PLOTS_DRIVE}/{MODEL_NAME}_training_curves.pdf")
else:
    print("  ⚠️  Skipped — no history available")

# ── 8b. Confusion matrix (raw + normalised) ──────────────────────────────────
print("\n[2/5] Confusion matrix (raw)...")
plot_confusion_matrix(y_true, y_pred, CLASS_NAMES, MODEL_NAME,
                      PLOTS_DRIVE, REPORT_DRIVE, normalize=False)
print(f"  ✓ {MODEL_NAME}_confusion_matrix.png")

print("\n[3/5] Confusion matrix (normalised)...")
plot_confusion_matrix(y_true, y_pred, CLASS_NAMES, MODEL_NAME,
                      PLOTS_DRIVE, REPORT_DRIVE, normalize=True)
print(f"  ✓ {MODEL_NAME}_confusion_matrix_normalized.png")

# ── 8c. Per-class precision / recall / F1 ───────────────────────────────────
print("\n[4/5] Per-class metrics chart...")
plot_per_class_metrics(metrics, MODEL_NAME, PLOTS_DRIVE, REPORT_DRIVE)
print(f"  ✓ {MODEL_NAME}_per_class_metrics.png")

# ── 8d. Top-5 misclassified samples ─────────────────────────────────────────
print("\n[5/5] Top-5 misclassified samples...")
plot_top_misclassified(
    test_loader.dataset, y_true, y_pred, y_prob,
    CLASS_NAMES, MODEL_NAME, PLOTS_DRIVE, REPORT_DRIVE, top_k=5,
)
print(f"  ✓ {MODEL_NAME}_top_misclassified.png")

# ── 9. Save metrics JSON + classification report ─────────────────────────────
print("\nSaving metrics and report...")
write_json(metrics, METRICS_DRIVE / f"{MODEL_NAME}_test_metrics.json")
save_classification_report(
    y_true, y_pred, CLASS_NAMES,
    METRICS_DRIVE / f"{MODEL_NAME}_classification_report.txt",
)
np.savez(
    METRICS_DRIVE / f"{MODEL_NAME}_predictions.npz",
    y_true=y_true, y_pred=y_pred, y_prob=y_prob,
)

# ── 10. Summary ──────────────────────────────────────────────────────────────
print("\n" + "═"*60)
print(f"  ALL OUTPUTS SAVED TO DRIVE")
print("═"*60)
print(f"  Plots    → {PLOTS_DRIVE}")
print(f"  Reports  → {REPORT_DRIVE}")
print(f"  Metrics  → {METRICS_DRIVE}")
print()
print("  Files generated:")
files = [
    f"{MODEL_NAME}_training_curves.png / .pdf",
    f"{MODEL_NAME}_confusion_matrix.png / .pdf",
    f"{MODEL_NAME}_confusion_matrix_normalized.png / .pdf",
    f"{MODEL_NAME}_per_class_metrics.png / .pdf",
    f"{MODEL_NAME}_top_misclassified.png / .pdf",
    f"{MODEL_NAME}_test_metrics.json",
    f"{MODEL_NAME}_classification_report.txt",
    f"{MODEL_NAME}_predictions.npz",
]
for f in files:
    print(f"  ✓ {f}")
print("═"*60)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Device : cuda
GPU    : NVIDIA A100-SXM4-40GB (42.4 GB)

Checkpoint : /content/drive/MyDrive/pk_politicians_results/checkpoints/efficientnet_b2_best.pth
Exists     : True

Building model architecture...
  efficientnet_b2  |  7.72M params
✓ Weights loaded from Drive checkpoint

Loading test dataset...
  5 test batches  |  159 test images  |  16 classes

Running inference on test set...

── Test metrics for efficientnet_b2 ──────────────────
  accuracy        : 0.8553
  macro precision : 0.8638
  macro recall    : 0.8592
  macro F1        : 0.8540
  weighted F1     : 0.8524
────────────────────────────────────────────────────

✓ Training history loaded  (27 epochs)

[1/5] Training curves...
  ✓ PNG : /content/drive/MyDrive/pk_politicians_results/plots/efficientnet_b2_training_curves.png
  ✓ PDF : /content/drive/MyDrive/pk_politicians_results/plots/efficientnet_b

## 9. (Manual) commit results to GitHub

After this notebook completes, commit the new files in your local repo:

```bash
cd Image-Classification-with-CNN
git add results/metrics/ results/plots/ report/figures/
git commit -m "feat: training run for {MODEL}"
git push origin main
```